# Import necessary functions

In [1]:
import zipfile
import shutil
import os
import stat
import xml.etree.ElementTree as ET
from pathlib import Path
from math import ceil

# Input the folder containing the presentation without embedded sheets

Create the input and output folders in the same working directory and type their name in the provision given below.

In [ ]:
input_dir = Path("input_pptx") 
output_dir = Path("output_pptx") 
temp_dir = Path("pptx_temp")

input_dir.mkdir(exist_ok=True)
output_dir.mkdir(exist_ok=True)

# Python Script for Converting Presentation

In [ ]:
ns = {
    'c': "http://schemas.openxmlformats.org/drawingml/2006/chart",
    'a': "http://schemas.openxmlformats.org/drawingml/2006/main",
    'r': "http://schemas.openxmlformats.org/officeDocument/2006/relationships"
}

# Register namespaces to avoid PowerPoint corrupting prefixes:
for prefix, uri in ns.items():
    ET.register_namespace(prefix, uri)

def on_rm_error(func, path, exc_info):
    """Handle read-only files safely during cleanup."""
    try:
        os.chmod(path, stat.S_IWRITE)
        func(path)
    except Exception:
        pass # Suppress failures if file is strictly locked
    
def repackage_pptx(temp_dir: Path, output_path: Path):
    """Re-zip the temp_dir into a valid .pptx with proper internal structure."""
    with zipfile.ZipFile(output_path, 'w', zipfile.ZIP_DEFLATED) as pptx:
        for folder, _, files in os.walk(temp_dir):
            for fname in files:
                full_path = Path(folder) / fname
                arcname = full_path.relative_to(temp_dir).as_posix()
                pptx.write(full_path, arcname)

def col_letter(n):
    """Convert 0-based column index to Excel letter (e.g., 0 -> A, 26 -> AA)"""
    result = ''
    while n >= 0:
        result = chr(n % 26 + 65) + result
        n = n // 26 - 1
    return result

def detect_shape_and_generate_formula(pt_count, col_count, is_str=True):
    """Generate Excel formula range based on number of points and columns"""
    start_row = 2
    end_row = start_row + pt_count - 1
    start_col = 0 if is_str else 1 
    end_col = start_col + col_count - 1
    return f"Sheet1!${col_letter(start_col)}${start_row}:${col_letter(end_col)}${end_row}"

def convert_lit_to_ref(parent_elem, tag_name):
    """Safely converts literal XML chart tags to reference XML chart tags."""
    lit_tag = 'strLit' if tag_name == 'cat' else 'numLit'
    ref_tag = 'strRef' if tag_name == 'cat' else 'numRef'
    cache_tag = 'strCache' if tag_name == 'cat' else 'numCache'

    old_tag_elem = parent_elem.find(f'c:{tag_name}', ns)
    if old_tag_elem is None:
        return

    lit_elem = old_tag_elem.find(f'c:{lit_tag}', ns)
    if lit_elem is None:
        return # Already a reference or empty

    pt_list = lit_elem.findall('c:pt', ns)
    pt_count = len(pt_list)
    pt_count_elem = lit_elem.find('c:ptCount', ns)
    col_count = 1
    
    for pt in pt_list:
        v = pt.find('c:v', ns)
        if v is not None and v.text and "," in v.text:
            col_count = len(v.text.split(","))
            break

    cell_formula = detect_shape_and_generate_formula(pt_count, col_count, is_str=(tag_name == 'cat'))

    # Generate clean standard OpenXML compliant tags
    ref_elem = ET.Element(f"{{{ns['c']}}}{ref_tag}")
    f = ET.SubElement(ref_elem, f"{{{ns['c']}}}f")
    f.text = cell_formula

    cache_elem = ET.SubElement(ref_elem, f"{{{ns['c']}}}{cache_tag}")
    if pt_count_elem is not None:
        cache_elem.append(pt_count_elem)
    else:
        new_pt_count = ET.SubElement(cache_elem, f"{{{ns['c']}}}ptCount")
        new_pt_count.set('val', str(pt_count))

    for pt in pt_list:
        cache_elem.append(pt)

    # Clean replace
    parent_elem.remove(old_tag_elem)
    new_tag = ET.Element(f"{{{ns['c']}}}{tag_name}")
    new_tag.append(ref_elem)
    parent_elem.append(new_tag)

def process_chart(chart_file):
    tree = ET.parse(chart_file)
    root = tree.getroot()
    modified = False

    # Strip out external data links if they exist
    external = root.find('.//c:externalData', ns)
    if external is not None:
        root.remove(external)
        modified = True

    # Loop through ALL series blocks within the chart to map multi-series data sets
    for ser in root.findall('.//c:ser', ns):
        convert_lit_to_ref(ser, 'cat')
        convert_lit_to_ref(ser, 'val')
        modified = True

    if modified:
        print(f"Processing changes in {chart_file.name}")
        tree.write(chart_file, encoding='utf-8', xml_declaration=True)

def process_pptx_file(input_path: Path, output_path: Path):
    if temp_dir.exists():
        shutil.rmtree(temp_dir, onerror=on_rm_error)
    temp_dir.mkdir(parents=True, exist_ok=True)

    try:
        with zipfile.ZipFile(input_path, 'r') as zip_ref:
            zip_ref.extractall(temp_dir)
    except zipfile.BadZipFile:
        print(f"❌ Skipped invalid or corrupted file: {input_path.name}")
        return

    charts_dir = temp_dir / "ppt" / "charts"
    if charts_dir.exists():
        for chart_file in charts_dir.glob("chart*.xml"):
            process_chart(chart_file)

    repackage_pptx(temp_dir, output_path)
    shutil.rmtree(temp_dir, onerror=on_rm_error)

# Batch process all pptx documents cleanly
for pptx_file in input_dir.glob("*.pptx"):
    output_file = output_dir / pptx_file.name
    process_pptx_file(pptx_file, output_file)

print("✅ All PPTX files processed seamlessly.")